In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/bahaa1515/EECE-693-project.git"
COLAB_REPO_DIR = Path("/content/EECE-693-project")

def find_project_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "src" / "config.py").exists():
            return candidate
    if COLAB_REPO_DIR.exists() and (COLAB_REPO_DIR / "src" / "config.py").exists():
        return COLAB_REPO_DIR
    try:
        import google.colab  # type: ignore  # noqa: F401
    except ImportError as exc:
        raise FileNotFoundError(
            "Could not find the project root. Run this notebook from the repo root, "
            "from the notebooks folder, or clone the repo in Colab first."
        ) from exc
    subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)
    return COLAB_REPO_DIR

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import SMARTWATCH_FILES, PATIENT_INFO_FILE, WEEKLY_FILE, OUTPUT_TABLES


In [3]:
patient_info = pd.read_csv(PATIENT_INFO_FILE)
weekly = pd.read_csv(WEEKLY_FILE)

print("patient_info shape:", patient_info.shape)
print("weekly shape:", weekly.shape)

display(patient_info.head())
display(weekly.head())

patient_info shape: (22, 28)
weekly shape: (324, 12)


,user_key,sex,age_range,bmi_range,age_diagnosed_range,max_pef_expected,smoker,pack_years,race,severity,...,position_start_date,pef_start_date,inhaler_start_date,daily_end_date,weekly_end_date,miband_end_date,position_end_date,pef_end_date,inhaler_end_date,phase2_end
0,113,female,30-39yo,Obesity,7-11yo,485,Never (<100 cigarettes in lifetime),0.00,White,Severe,...,1,1.0,0.0,183,183,183.0,183,40.0,177.0,183
1,190,male,50+yo,Pre-obesity,19+yo,575,Previous,18.75,White,Severe,...,1,6.0,1.0,183,182,148.0,183,183.0,19.0,183
2,217,female,30-39yo,Obesity,19+yo,495,Previous,6.30,White,Moderate,...,0,0.0,1.0,1,0,1.0,1,0.0,1.0,1
3,278,female,18-29yo,Pre-obesity,12-18yo,495,Previous,1.20,White,Moderate,...,0,NaN,-20.0,15,15,15.0,13,NaN,106.0,106
4,294,female,50+yo,Obesity,0-6yo,455,Never (<100 cigarettes in lifetime),0.00,White,Moderate,...,0,0.0,-5.0,183,182,183.0,183,183.0,184.0,184


,user_key,date,weekly_night_symp,weekly_day_symp,weekly_limit_activity,weekly_short_breath,weekly_wheeze,weekly_relief_inhaler,weekly_doc,weekly_hospital,weekly_er,weekly_oral
0,113,1,2.0,3.0,1.0,2,2,3,-4,0,0,2.0
1,113,7,2.0,3.0,2.0,2,2,2,-5,0,0,2.0
2,113,14,5.0,5.0,5.0,4,4,5,"-1,-3",0,-2,3.0
3,113,22,4.0,5.0,4.0,4,2,4,-3,0,0,3.0
4,113,28,3.0,3.0,3.0,3,2,3,0,0,0,3.0


In [4]:
for f in SMARTWATCH_FILES:
    df = pd.read_csv(f)
    print("=" * 80)
    print(f.name)
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print(df.head())
    print(df.isna().sum())

anonym_aamos00_smartwatch1.csv
shape: (1000000, 7)
columns: ['user_key', 'date', 'time', 'activity_type', 'intensity', 'steps', 'hr']
   user_key  date      time  activity_type  intensity  steps    hr
0       113     0  09:35:00             80         31      0   NaN
1       113     0  09:36:00             96          6      0  72.0
2       113     0  09:37:00             80          0      0  73.0
3       113     0  09:38:00             80         13      0  74.0
4       113     0  09:39:00             90         20      0  75.0
user_key              0
date                  0
time                  0
activity_type         0
intensity             0
steps                 0
hr               197760
dtype: int64
anonym_aamos00_smartwatch2.csv
shape: (1000000, 7)
columns: ['user_key', 'date', 'time', 'activity_type', 'intensity', 'steps', 'hr']
   user_key  date      time  activity_type  intensity  steps    hr
0       447    70  06:24:00             80         70      0  91.0
1       447    

In [5]:
usecols = ["user_key", "date", "time", "activity_type", "intensity", "steps", "hr"]

smartwatch_parts = [pd.read_csv(f, usecols=usecols) for f in SMARTWATCH_FILES]
smartwatch = pd.concat(smartwatch_parts, ignore_index=True)

print("combined shape:", smartwatch.shape)
display(smartwatch.head())

combined shape: (2101829, 7)


,user_key,date,time,activity_type,intensity,steps,hr
0,113,0,09:35:00,80,31,0,NaN
1,113,0,09:36:00,96,6,0,72.0
2,113,0,09:37:00,80,0,0,73.0
3,113,0,09:38:00,80,13,0,74.0
4,113,0,09:39:00,90,20,0,75.0


In [6]:
summary = pd.DataFrame({
    "dtype": smartwatch.dtypes.astype(str),
    "missing_count": smartwatch.isna().sum(),
    "missing_pct": (smartwatch.isna().mean() * 100).round(2),
    "n_unique": smartwatch.nunique(dropna=True)
})

display(summary)
summary.to_csv(OUTPUT_TABLES / "smartwatch_column_summary.csv")

,dtype,missing_count,missing_pct,n_unique
user_key,int64,0,0.0,20
date,int64,0,0.0,185
time,str,0,0.0,1440
activity_type,int64,0,0.0,39
intensity,int64,0,0.0,256
steps,int64,0,0.0,178
hr,float64,531728,25.3,153


In [7]:
user_summary = (
    smartwatch.groupby("user_key")
    .agg(
        n_rows=("user_key", "size"),
        min_date=("date", "min"),
        max_date=("date", "max"),
        missing_hr_pct=("hr", lambda s: round(s.isna().mean() * 100, 2)),
        missing_steps_pct=("steps", lambda s: round(s.isna().mean() * 100, 2)),
    )
    .reset_index()
)

display(user_summary.head())
user_summary.to_csv(OUTPUT_TABLES / "smartwatch_user_summary.csv", index=False)

,user_key,n_rows,min_date,max_date,missing_hr_pct,missing_steps_pct
0,113,240599,0,183,19.18,0.0
1,190,181965,1,148,18.06,0.0
2,217,918,0,1,24.84,0.0
3,278,7188,10,15,20.21,0.0
4,294,224323,-1,183,18.98,0.0


In [8]:
print("Negative dates:", (smartwatch["date"] < 0).sum())
print("Negative steps:", (smartwatch["steps"] < 0).sum())
print("hr <= 0:", (smartwatch["hr"] <= 0).sum())
print("Duplicate rows:", smartwatch.duplicated().sum())

Negative dates: 70
Negative steps: 0
hr <= 0: 0
Duplicate rows: 0


In [9]:
print("\n=== SMARTWATCH USER COUNT ===")
n_unique_users = smartwatch["user_key"].nunique()
print(f"Number of unique users in smartwatch data: {n_unique_users}")
print(f"\nUnique user_key values:")
print(sorted(smartwatch["user_key"].unique()))


=== SMARTWATCH USER COUNT ===
Number of unique users in smartwatch data: 20

Unique user_key values:
[np.int64(113), np.int64(190), np.int64(217), np.int64(278), np.int64(294), np.int64(328), np.int64(343), np.int64(398), np.int64(447), np.int64(454), np.int64(473), np.int64(514), np.int64(625), np.int64(701), np.int64(702), np.int64(748), np.int64(764), np.int64(808), np.int64(917), np.int64(939)]
